In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\Kajal\Downloads\100_Unique_QA_Dataset.csv")

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()

tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [5]:
#vocab
vocab = {'<UNK>':0}

In [6]:
def build_vocab(df):
    tokenized_question = tokenize(df['question'])
    tokenized_answer = tokenize(df['answer'])

    merge_tokens = tokenized_question + tokenized_answer
    
    for token in merge_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [7]:
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [8]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [9]:
len(vocab)

324

In [10]:
#convert words to numerical
def text_to_indices(text,vocab):

    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text       
        

In [11]:
text_to_indices("what is campusx",vocab)

[1, 2, 0]

In [12]:
import torch
from torch.utils.data import Dataset,DataLoader

In [13]:
class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab
        
    def __len__(self):
        return self.df.shape[0]
        
    def __getitem__(self,index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)

        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [14]:
dataset = QADataset(df,vocab)

In [15]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [16]:
for question,answer in dataloader:
    print(question,answer)

tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([[128]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([[110]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([[106]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([[74]])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([[124]])
tensor([[ 1,  2,  3, 17, 18, 19, 20, 21, 22]]) tensor([[23]])
tensor([[ 10,  96,   3, 104, 239]]) tensor([[240]])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([[121]])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor

In [17]:
import torch.nn as nn

In [63]:
class SimpleRNN(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.RNN(50,64,batch_first=True)
        self.fc = nn.Linear(64,vocab_size)
        
    def forward(self,question):
        embedded_question = self.embedding(question)
        hidden,final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output
    

In [64]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [65]:
x = nn.Embedding(324,embedding_dim=50)

In [66]:
x(dataset[0][0])

tensor([[ 1.7424e+00,  7.7223e-01, -8.7094e-01,  1.3490e+00,  4.6425e-01,
         -1.0642e+00, -5.2405e-01,  1.0852e-01, -3.3624e-01,  1.2553e+00,
         -8.4740e-02,  8.8188e-01,  2.4854e-01, -4.8307e-01, -5.6147e-01,
          1.2682e+00,  7.0251e-02, -5.5475e-01,  1.8029e+00, -1.9357e-02,
         -2.0660e-01,  5.2656e-01, -1.9183e-02, -6.9548e-01,  6.8837e-01,
         -5.5452e-01, -1.8347e+00, -6.1083e-01, -1.6503e-02, -4.9524e-01,
          5.3884e-01, -3.6289e-01,  1.4053e+00,  1.0949e+00,  3.7359e-01,
          1.0089e+00,  7.5652e-01, -1.7162e-01, -1.4575e+00, -3.6895e-01,
          5.5416e-01,  6.0058e-01,  1.2683e+00,  5.4400e-01,  3.6318e-01,
          4.3156e-01, -1.5605e+00,  2.7205e+00, -1.1455e+00,  1.1272e+00],
        [ 7.3934e-01, -1.9251e+00, -1.5171e-01, -1.1654e+00,  7.9922e-01,
          3.0898e-01, -4.0087e-02,  2.6435e-01, -3.0606e+00, -1.2344e+00,
          9.5075e-01,  1.3322e+00,  9.1770e-01, -2.1636e-01,  8.6313e-01,
          1.3436e+00,  8.7073e-01, -1

In [67]:
y = nn.RNN(50,64)

In [68]:
b = x(dataset[0][0])

In [69]:
b

tensor([[ 1.7424e+00,  7.7223e-01, -8.7094e-01,  1.3490e+00,  4.6425e-01,
         -1.0642e+00, -5.2405e-01,  1.0852e-01, -3.3624e-01,  1.2553e+00,
         -8.4740e-02,  8.8188e-01,  2.4854e-01, -4.8307e-01, -5.6147e-01,
          1.2682e+00,  7.0251e-02, -5.5475e-01,  1.8029e+00, -1.9357e-02,
         -2.0660e-01,  5.2656e-01, -1.9183e-02, -6.9548e-01,  6.8837e-01,
         -5.5452e-01, -1.8347e+00, -6.1083e-01, -1.6503e-02, -4.9524e-01,
          5.3884e-01, -3.6289e-01,  1.4053e+00,  1.0949e+00,  3.7359e-01,
          1.0089e+00,  7.5652e-01, -1.7162e-01, -1.4575e+00, -3.6895e-01,
          5.5416e-01,  6.0058e-01,  1.2683e+00,  5.4400e-01,  3.6318e-01,
          4.3156e-01, -1.5605e+00,  2.7205e+00, -1.1455e+00,  1.1272e+00],
        [ 7.3934e-01, -1.9251e+00, -1.5171e-01, -1.1654e+00,  7.9922e-01,
          3.0898e-01, -4.0087e-02,  2.6435e-01, -3.0606e+00, -1.2344e+00,
          9.5075e-01,  1.3322e+00,  9.1770e-01, -2.1636e-01,  8.6313e-01,
          1.3436e+00,  8.7073e-01, -1

In [70]:
y(b)[0].shape

torch.Size([6, 64])

In [71]:
y(b)[1]

tensor([[-0.5667,  0.0654,  0.3075,  0.9135, -0.2130,  0.1003,  0.3911,  0.5513,
         -0.0705,  0.3262,  0.1112, -0.4539,  0.4510,  0.0048, -0.4952,  0.6794,
          0.5853,  0.2159, -0.6067,  0.3517, -0.6149,  0.0677, -0.4369,  0.1456,
          0.6704,  0.1443,  0.2864,  0.0089,  0.5397,  0.0327,  0.2996,  0.7436,
          0.1039,  0.6277, -0.6685,  0.1457,  0.2095,  0.0762, -0.0261, -0.3378,
          0.5047,  0.2577, -0.0867, -0.2630,  0.5314, -0.0606, -0.2477,  0.5697,
         -0.0216,  0.1549, -0.8286, -0.1782,  0.6383, -0.6955, -0.5070, -0.7128,
          0.3513,  0.4012,  0.5077,  0.4826, -0.8327, -0.4634,  0.4488,  0.4015]],
       grad_fn=<SqueezeBackward1>)

In [72]:
z = nn.Linear(64,324)

In [73]:
a = y(b)[1]

In [74]:
z(a).shape

torch.Size([1, 324])

In [75]:
learning_rate = 0.001
epochs = 20

In [76]:
model = SimpleRNN(len(vocab))

In [77]:
criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [80]:
#training loop
for epoch in range(epochs):
    total_loss = 0
    for question,answer in dataloader:
        optimizer.zero_grad()

        #forward pass
        output = model(question)
        #loss
        loss = criteria(output,answer[0])
        #gradient
        loss.backward()
        #update
        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch {epoch+1},Loss:{total_loss:4f}") 

Epoch 1,Loss:525.606074
Epoch 2,Loss:461.069023
Epoch 3,Loss:383.936894
Epoch 4,Loss:321.218496
Epoch 5,Loss:269.353949
Epoch 6,Loss:220.877927
Epoch 7,Loss:176.179850
Epoch 8,Loss:136.573067
Epoch 9,Loss:104.780779
Epoch 10,Loss:79.551646
Epoch 11,Loss:61.182955
Epoch 12,Loss:47.195227
Epoch 13,Loss:37.341325
Epoch 14,Loss:29.983415
Epoch 15,Loss:24.664609
Epoch 16,Loss:20.539725
Epoch 17,Loss:17.255001
Epoch 18,Loss:14.673384
Epoch 19,Loss:12.562295
Epoch 20,Loss:10.827111


In [79]:
print(output.shape)
print(answer.shape)

torch.Size([1, 324])
torch.Size([1, 1])


In [104]:
def predict(model,question,threshold=0.5):

    #convert question to number 
    numerical_question = text_to_indices(question,vocab)
    #tensor
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    #send to model
    output = model(question_tensor)
    #convert logits to prob
    probs = torch.nn.functional.softmax(output,dim=1)
    #find index of max prob
    value,index = torch.max(probs,dim=1)
    print(value,index)

    if value < threshold:
        print("I don`t know")
    print(list(vocab.keys())[index])

In [105]:
predict(model,'what is the capital of france')

tensor([0.8531], grad_fn=<MaxBackward0>) tensor([7])
paris


In [106]:
torch.save(model.state_dict(), "model.pth")

In [107]:
import pickle

with open("vocab.pkl","wb") as f:
    pickle.dump(vocab,f)

In [108]:
reverse_vocab = {}

for word,index in vocab.items():
    reverse_vocab[index] = word

In [109]:
with open("reverse_vocab.pkl","wb") as f:
    pickle.dump(reverse_vocab,f)